# Plot spot position as time series data
env : data_vis_32

# 1.0 Import relevant packages

In [2]:
# import pypyodbc
import pandas as pd
import plotly.express as px
import seaborn as sns
from matplotlib.colors import to_hex
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

# 2.0 Import spot position and size QA data

In [3]:
data_path = r"../data/xlsx_exported_from_access/SpotPositionResults.xlsx"

df = pd.read_excel(data_path)

df.head(2)



,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,hor_rt_gradient,hor_lt_gradient,hor_fwhm,vert_rt_gradient,vert_lt_gradient,vert_fwhm,bltr_rt_gradient,bltr_lt_gradient,bltr_fwhm,tlbr_rt_gradient,tlbr_lt_gradient,tlbr_fwhm
0,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Centre,-0.2357,124.9224,-9.059840,9.157258,13.342140,-8.964427,9.333333,13.536155,-8.485281,8.747554,14.216962,-8.909545,8.992812,13.842831
1,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Left,-125.0153,125.4501,-8.871094,9.120482,13.562307,-8.964427,9.333333,13.712522,-8.747554,8.591347,14.310494,-8.747554,8.992812,13.842831


# 3.0 exploratory data analysis - understand your data

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41172 entries, 0 to 41171
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ADate             41172 non-null  datetime64[ns]
 1   MachineName       41172 non-null  object        
 2   Energy            41172 non-null  int64         
 3   Device            41172 non-null  object        
 4   Gantry Angle      41172 non-null  int64         
 5   Spot              41172 non-null  object        
 6   x-pos             41172 non-null  float64       
 7   y-pos             41172 non-null  float64       
 8   hor_rt_gradient   41172 non-null  float64       
 9   hor_lt_gradient   41172 non-null  float64       
 10  hor_fwhm          41172 non-null  float64       
 11  vert_rt_gradient  41172 non-null  float64       
 12  vert_lt_gradient  41172 non-null  float64       
 13  vert_fwhm         41172 non-null  float64       
 14  bltr_rt_gradient  4117

# 4.0 filtering ABS Shift data

In [5]:
sub_df = df[["ADate",	"MachineName", 	"Energy", "Device", "Gantry Angle", "Spot", "x-pos", "y-pos"]].copy()

In [6]:
pred_xrv4000 = {'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175], \
                'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125], \
                'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]}

sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

sub_df['abs_xpos'] = sub_df["x-pos"] - sub_df["px_pos"]
sub_df['abs_ypos'] = sub_df["y-pos"] - sub_df["py_pos"]

In [7]:
start_date = pd.Timestamp.today() - pd.DateOffset(months=12)
selected_df = sub_df[(df["MachineName"]=="Gantry 2") & (df["Device"] == "XRV-3000") & (df['ADate'] >= start_date)].copy()
# Calculate average abs_xpos per adate and energy
selected_df['avg_abs_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])["abs_xpos"].transform('mean')

selected_df.head(5)

,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,px_pos,py_pos,abs_xpos,abs_ypos,avg_abs_pos
37618,2025-10-08 16:45:32,Gantry 2,70,XRV-3000,0,Bottom-Centre,-0.2267,125.1980,0,125,-0.2267,0.1980,-0.300178
37619,2025-10-08 16:45:32,Gantry 2,70,XRV-3000,0,Bottom-Left,-125.3861,125.1288,-125,125,-0.3861,0.1288,-0.300178
37620,2025-10-08 16:45:32,Gantry 2,70,XRV-3000,0,Bottom-Right,125.0294,125.0969,125,125,0.0294,0.0969,-0.300178
37621,2025-10-08 16:45:32,Gantry 2,70,XRV-3000,0,Centre,-0.3810,0.1202,0,0,-0.3810,0.1202,-0.300178
37622,2025-10-08 16:45:32,Gantry 2,70,XRV-3000,0,Left,-125.5507,0.4636,-125,0,-0.5507,0.4636,-0.300178


### Figure is for visualisation purposes only. Drop down and date - slider filters not currently functional.

In [8]:
# Plotting Function
def plotly_spot_position_Filtered_XY(df, pos, gantry, device, energy, gantry_angle, n_months):
    """ plot spot position time series data
        df = spot position dataframe
     """
    
    # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)
    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) &(df["Gantry Angle"] == gantry_angle)]
    

    # set colour
    palette = sns.color_palette("deep", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]

    # Plot y_pos figure
    fig = px.scatter(
        selected_df,
        x='ADate',
        y="abs_ypos",
        symbol='Spot', 
        color='Spot', # hue
        color_discrete_sequence= px.colors.qualitative.T10,
        title='Absolute shift',
        labels={'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")

    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))
    
    # Update to include drop down box for filtering
    # Functions to enable automatically updating graph filtering from drop-down menus
    def visible_for_g(target_g): # Gantry selection
        return [
            True if g == target_g else "legendonly"
            for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]
        ]
    
    def visible_for_ga(target_ga): # Gantry Angle selection
        return [
            True if ga == target_ga else "legendonly"
            for ga in ["0", "90", "180", "270"]
        ]
    
    def visible_for_e(target_e): # Energy selection
        return [
            True if e == target_e else "legendonly"
            for e in ["70", "100", "150", "200", "270"]
        ]

    def visible_for_pos(target_pos): # Position selection
        return [
            True if pos == target_pos else "legendonly"
            for pos in ["abs_xpos", "abs_ypos"]
        ]

    # Define Gantry Drop-down menu
    g_button = []
    for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]:
            g_button.append(
                dict(
                    label=f"{g}",
                    method="update",
                    args=[{"visible": visible_for_g(g)}]
                    )
            )

    # Define Gantry Angle drop-down menu
    ga_button = []
    for ga in ["0", "90", "180", "270"]:
            ga_button.append(
                dict(
                    label=f"{ga}",
                    method="update",
                    args=[{"visible": visible_for_ga(ga)}]
                    )
            )

    # Define Energy drop-down menu
    e_button = []
    for e in ["70", "100", "150", "200", "270"]:
            e_button.append(
                dict(
                    label=f"{e} MeV",
                    method="update",
                    args=[{"visible": visible_for_e(e)}]
                    )
            )

    # Define x/y_pos drop-down menu
    pos_button = []
    for pos in ["abs_xpos", "abs_ypos"]:
            pos_button.append(
                dict(
                    label=f"{pos}",
                    method="update",
                    args=[{"visible": visible_for_pos(pos)}]
                    )
            )

    fig.update_layout(
        updatemenus=[
            # Update graph to reflect Gantry drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.02,
                y=1.14,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=g_button
            ),

            # Update graph to reflect Gantry Angle drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.16,
                y=1.14,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=ga_button
            ),

            # Update graph to reflect Energy drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.28,
                y=1.14,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=e_button
            ),

            # Update graph to reflect Position drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.43,
                y=1.14,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=pos_button
            ),
        ]
    ),

    fig.update_layout(
        annotations=[
            dict(text="Gantry", x=0.0, xref="paper", y=1.12, yref="paper", align = "left", showarrow=False),
            dict(text="Angle:", x=0.12, xref="paper", y=1.12, yref="paper", align="left", showarrow=False),
            dict(text="Energy:", x=0.23, xref="paper", y=1.12, yref="paper", align="left", showarrow=False),
            dict(text="Position:", x=0.40, xref="paper", y=1.1, yref="paper", align="left", showarrow=False)
        ])
    

    # Create and add slider
    months = []
    for i in range(n_months):
        month = dict(
            method="update",
            args=[{"visible": [True] * len(fig.data)}  # layout attribute
        ])
        month["args"][0]["visible"][i] = True  # Toggle i'th trace to "visible"
        months.append(month)

    # Define sliders for date range selection
    sliders = [dict(
    active=10,
    currentvalue={"prefix": "Previous "},
    pad={"t": 50},
    steps=months
    )]

    fig.update_layout(
        sliders=sliders
    )

    i=0
    for step in fig.layout.sliders[0].steps:
        step['label'] = f"{i} Months"
        i += 1

    # Show Plot
    fig.show()
    return 


In [9]:
# plotting absolute both x-pos/y-pos shift, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last2 months
plotly_spot_position_Filtered_XY(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 12)

# Spatial Grid Subplot

### Figure is for visualisation purposes only. Drop down and date - slider filters not currently functional.

In [10]:
pred_xrv4000 = {'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175], \
                'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125], \
                'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]}

pred_xrv3000 = {'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125]}

markers_plotly = {240: 'circle-open', 200: 'triangle-left', 150:'square', 100:'x-open', 70:'triangle-up'}

In [11]:

def plot_spot_grid_plotly_sbe(df, gantry, device, tolerance, n_months):
    """ plot the data in grids. """

    # 1. Data Preparation
    df['ADate'] = pd.to_datetime(df['ADate'])
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)
    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date)].copy()

    if selected_df.empty:
        print(f"No data found for {gantry} using {device} in the last {n_months} months.")
        return None

    # 2. Dynamic Color Palette (HLS)
    # Sorting ensures the color gradient follows the energy levels
    ens = sorted(list(selected_df["Energy"].unique()), reverse=True)
    hls_palette = sns.color_palette("hls", len(ens)).as_hex()
    colours = dict(zip(ens, hls_palette))

    # 3. Plot Configuration
    if tolerance == 1:
        b = 2.5
        title = f"Relative spot positions ({tolerance} mm tolerance)"
    else:
        b = 5
        title = f"Absolute spot positions ({tolerance} mm tolerance)"
        
    if device == 'XRV-4000':
        pos = pred_xrv4000
        nrows, ncols = 5, 3
    elif device == 'XRV-3000':
        pos = pred_xrv3000
        nrows, ncols = 3, 3
    
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=list(pos.keys()),
                        horizontal_spacing=0.05, vertical_spacing=0.07,
                        shared_xaxes=True,  # All subplots in a column share the same X range
                        shared_yaxes=True)   # All subplots in a row share the same Y range)

    keys = list(pos.keys())
    all_shapes = []  
    plotted_energies = set() # Track for clean legend
    planned_legend_added = False
    measured_legend_added = False
    
    # 4. Add filtering from drop down boxes and month slider
    # Functions to enable automatically updating graph filtering from drop-down menus
    def visible_for_g(target_g): # Gantry selection
        return [
            True if g == target_g else "legendonly"
            for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]
        ]
    
    def visible_for_ga(target_ga): # Gantry Angle selection
        return [
            True if ga == target_ga else "legendonly"
            for ga in ["0", "90", "180", "270"]
        ]
    
    def visible_for_e(target_e): # Energy selection
        return [
            True if e == target_e else "legendonly"
            for e in ["70", "100", "150", "200", "270"]
        ]

    # Define Gantry Drop-down menu
    g_button = []
    for g in ["Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4"]:
            g_button.append(
                dict(
                    label=f"{g}",
                    method="update",
                    args=[{"visible": visible_for_g(g)}]
                    )
            )

    # Define Gantry Angle drop-down menu
    ga_button = []
    for ga in ["0", "90", "180", "270"]:
            ga_button.append(
                dict(
                    label=f"{ga}",
                    method="update",
                    args=[{"visible": visible_for_ga(ga)}]
                    )
            )

    # Define Energy drop-down menu
    e_button = []
    for e in ["70", "100", "150", "200", "270"]:
            e_button.append(
                dict(
                    label=f"{e} MeV",
                    method="update",
                    args=[{"visible": visible_for_e(e)}]
                    )
            )

    fig.update_layout(
        updatemenus=[
            # Update graph to reflect Gantry drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.05,
                y=1.06,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=g_button
            ),

            # Update graph to reflect Gantry Angle drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.29,
                y=1.06,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=ga_button
            ),

            # Update graph to reflect Energy drop down box value
            dict(
                type="dropdown",
                direction="down",
                x=0.49,
                y=1.06,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=e_button
            )
        ]
    ),

    fig.update_layout(
        annotations=[
            dict(text="Gantry:", x=0.0, xref="paper", y=1.02, yref="paper", align = "left", showarrow=False),
            dict(text="Angle:", x=0.24, xref="paper", y=1.02, yref="paper", align="left", showarrow=False),
            dict(text="Energy:", x=0.44, xref="paper", y=1.02, yref="paper", align="left", showarrow=False)
        ])

    # 5. Grid Plotting Loop
    for i, p in enumerate(keys):
        row = i // ncols + 1
        col = i % ncols + 1
        cx, cy = pos[p]
        ndf = selected_df[selected_df['Spot'] == p].set_index('Energy')

        # Plot historical spot data
        for e in ndf.index:
            row_data = ndf.loc[[e]] 
            
            # Extract coordinates based on tolerance
            if tolerance == 2:
                x_vals = row_data['x-pos']
                y_vals = row_data['y-pos']
            else:
                x_vals = row_data['x-pos'] - row_data['centre_abs_xpos'] 
                y_vals = row_data['y-pos'] - row_data['centre_abs_ypos'] 
            
            # Format ADate for the hover label
            # We use .dt.strftime to make it look clean (e.g., 2022-04-21)
            date_strings = row_data['ADate'].dt.strftime('%Y-%m-%d %H:%M')

            # Logic for clean legend
            show_this_legend = False
            if e not in plotted_energies:
                show_this_legend = True
                plotted_energies.add(e)


            # Plot all data seperately, most recent = red cross then increasing transparency for previous results
            base_hex = colours.get(e, '#000000')
            fill_rgba = []; fill_rgba.append("rgba(256,0,0,1)");
            symbol_array = []; symbol_array.append("x");
            
            for n in range(1, x_vals.size):
                fill_rgba.append(f"rgba({','.join([str(int(base_hex[i:i+2], 16)) for i in (1, 3, 5)])}, {0.3* (1 - (n/x_vals.size))})")
                symbol_array.append(markers_plotly.get(e, 'circle'));

            # Add historical results
            fig.add_trace(
                go.Scatter(
                    x=x_vals.iloc[1:],
                    y=y_vals.iloc[1:],
                    mode='markers',
                    marker=dict(
                        symbol=symbol_array[1:],
                        size=7,
                        color=fill_rgba[1:],
                    ),
                    name=str(e),
                    legendgroup=str(e),
                    showlegend=show_this_legend,
                    customdata=date_strings,
                    hovertemplate=(
                        "<b>Energy: %{fullData.name} MeV</b><br>" +
                        "X: %{x:.2f}<br>" +
                        "Y: %{y:.2f}<br>" +
                        "Date: %{customdata}<extra></extra>"
                    )
                ),
                row=row, col=col,
            )

            # Add planned spot marker
            fig.add_trace(
                go.Scatter(
                    x=[cx], y=[cy],
                    mode='markers',
                    marker=dict(symbol='cross', color='black', size=5),
                    name='planned',
                    showlegend=not planned_legend_added
                ),
                row=row, col=col
            )
            planned_legend_added = True

            # Add most recent spot marker...
            fig.add_trace(
                go.Scatter(
                    x=x_vals.iloc[0:1],
                    y=y_vals.iloc[0:1],
                    mode='markers',
                    marker=dict(
                        symbol=symbol_array[0],
                        size=7,
                        color=fill_rgba[0],
                    ),
                    name = "Measured",
                    legendgroup = str(e),
                    showlegend = not measured_legend_added,
                    customdata=date_strings,
                    hovertemplate=(
                        "<b>Energy: %{fullData.name} MeV</b><br>" +
                        "X: %{x:.2f}<br>" +
                        "Y: %{y:.2f}<br>" +
                        "Date: %{customdata}<extra></extra>"
                    )
                ),
                row=row, col=col,
            )
            measured_legend_added = True

        # 6. Tolerance Shapes (Set to layer='below')
        all_shapes.extend([
            dict(
                type='circle',
                layer='below', 
                xref=f'x{i+1}', yref=f'y{i+1}',
                x0=cx - tolerance, y0=cy - tolerance,
                x1=cx + tolerance, y1=cy + tolerance,
                line=dict(color='#DBB40C'),
                fillcolor='#F5F5DC',
                opacity=0.5,
            ),
            dict(
                type='rect',
                layer='below',
                xref=f'x{i+1}', yref=f'y{i+1}',
                x0=cx - tolerance, y0=cy - tolerance,
                x1=cx + tolerance, y1=cy + tolerance,
                line=dict(color='#DBB40C', width=1)
            )
        ])
        
        # Invert Y-axis for radiotherapy coordinate systems
        fig.update_xaxes(range=[cx - b, cx + b], row=row, col=col)
        fig.update_yaxes(range=[cy + b, cy - b], scaleanchor=f"x{i+1}", row=row, col=col)

    # 7. Create and add slider
    months = []
    for i in range(n_months):
        month = dict(
            method="update",
            args=[{"visible": [True] * len(fig.data)}  # layout attribute
        ])
        month["args"][0]["visible"][i] = True  # Toggle i'th trace to "visible"
        months.append(month)

    # Define sliders for date range selection
    sliders = [dict(
    active=10,
    currentvalue={"prefix": "Previous "},
    pad={"t": 50},
    steps=months
    )]

    fig.update_layout(
        sliders=sliders
    )

    i=0
    for step in fig.layout.sliders[0].steps:
        step['label'] = f"{i} Months"
        i += 1
    
    # 8. Final Layout
    fig.update_layout(
        shapes=all_shapes,
        title=title,
        height=300 * nrows,
        width=300 * ncols,
        margin=dict(t=100, l=50, r=50, b=50),
        legend=dict(x=1.02, y=0.5, traceorder="normal"),
        template="plotly_white"
    )

    return fig

In [12]:
fig = plot_spot_grid_plotly_sbe(df = sub_df, gantry = "Gantry 1", device = "XRV-3000", tolerance = 2, n_months =12)

fig

# Generalise plotting function - plot spot position as a function of time
- Easier debugging
- better for unit testing
- Same filtered data can feed different plots : scatter plot | SPC chart | histogram |summary statistics table |control limits

This becomes especially important in Dash because a callback often creates several outputs from the same subset of data.

## Function: plot_by_spot_position()

In [26]:

def plot_by_spot_position(df, option, parameter, gantry, device, energy, gantry_angle, n_months, tolerance):
    """ plot spot position time series data

    ---input---
        df = dataframe
        option (str) = "spot_position" | "fwhm" | "spot_symmetry"
        gantry = "Gantry 1", "Gantry 2", 
        parameter = 
                    spot position option:  'abs_xpos', 'abs_ypos',  'rel_xpos', 'rel_ypos'
                    fwhm option: 'hor_fwhm', 'vert_fwhm', 'bltr_fwhm', 'tlbr_fwhm', "ave_fwhm"
                    spot_symmetry option : 'hor_rt_gradient', 'hor_lt_gradient', 
        device = "XRV-3000", "XRV-4000"
        energy = int,
        gantry_angle = 0,90,180,270
        n_month = int
        tolerence (float) = 2 (spot pos, abs) | fwhm from tps or baseline | None (gradient_ratio)

    ---output---
        fig
     """
    

    # valid the input before plotting
    valid_parameters = {
                        "spot_position": ["abs_xpos", "abs_ypos", "rel_xpos", "rel_ypos"],
                        "fwhm": ["hor_fwhm", "vert_fwhm", "bltr_fwhm", "tlbr_fwhm", "ave_fwhm"],
                        "spot_symmetry": ["gr_hor", "gr_vert", "gr_bltr", "gr_tlbr"]
                        }
    
    if option not in valid_parameters:
        raise ValueError( f"Invalid option '{option}'. "
                        f"Expected one of {list(valid_parameters.keys())}.")

    if parameter not in valid_parameters[option]:
        raise ValueError(f"Invalid parameter '{parameter}' for option '{option}'. "
                            f"Expected one of {valid_parameters[option]}.")
    
    #  # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) &(df["Gantry Angle"] == gantry_angle)]

    # set colour
    palette = sns.color_palette("hls", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]

    # define device
    device = pd.unique(selected_df["Device"])[0]

    y_axis_name = {'abs_xpos' : "Absolute X Position (mm)", 
                    'abs_ypos': "Absolute Y Position (mm)",
                    'rel_xpos': "Relative X Position (mm)", 
                    'rel_ypos' : "Relative Y Position (mm)", 
                    'hor_fwhm' : "Horizontal FWHM (mm)",
                    'vert_fwhm' : "Vertical FWHM (mm)", 
                    'bltr_fwhm' : "bltr FWHM (mm)",
                    'tlbr_fwhm' : "tlbr FWHM (mm)", 
                    'ave_fwhm' : "average FWHM (mm)", 
                    'gr_hor' : " Horizontal Gradient Ratio (RT/LT)", 
                    'gr_vert' : " Vertical Gradient Ratio (RT/LT)", 
                    'gr_bltr' : "BLTR Gradient Ratio (RT/LT)",
                    'gr_tlbr' : "TLBR Gradient Ratio (RT/LT)"}

    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y=parameter,
        symbol='Spot', 
        color='Spot',        # hue
        color_discrete_sequence= palette_hex,
        title=f'{gantry} - {device} - {y_axis_name[parameter]}',
        labels= {'ADate': 'Date',
                parameter: y_axis_name[parameter],
                'Spot': 'Spot'
                }, #ename axis titles, legend titles, and hover labels
        hover_data={'ADate': '|%Y-%m-%d'},
        height=500,
        opacity = 0.8
    )

    # Add tolerance bands +/- 2
    if "spot_position" in option:
        fig.add_hline(y=tolerance, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
        fig.add_hline(y=-tolerance, line_dash="dash", line_color="grey")
    
    elif "fwhm" in option:
        fig.add_hline(y=0.9*tolerance, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
        fig.add_hline(y=1.1*tolerance, line_dash="dash", line_color="grey")



    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size 
                                line=dict(width=2)),     # outline width
                                line=dict(width=1                # thinner connecting lines
                                ),
                    hovertemplate= "Date: %{x|%Y-%m-%d}<br>" +
                                    f"{parameter}: %{{y}}<br>" +
                                    "Spot: %{fullData.name}<extra></extra>")

    # Show plot
    fig.show()

    
    return 





## spot position data - calculate absolute and relative shift

In [14]:
sub_df = df[["ADate",	"MachineName", 	"Energy", "Device", "Gantry Angle", "Spot", "x-pos", "y-pos"]].copy()

sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

sub_df['abs_xpos'] = sub_df["x-pos"] - sub_df["px_pos"]
sub_df['abs_ypos'] = sub_df["y-pos"] - sub_df["py_pos"]

# create a dict with ADate as the key and abs x, y pos as the value
centre_spot_df = sub_df[sub_df["Spot"] == "Centre"]
centre_abs_xpos = (
                    centre_spot_df
                    .set_index(["ADate", "Energy"])["abs_xpos"]
                    .to_dict()
                    )

centre_abs_ypos = (
                    centre_spot_df
                    .set_index(["ADate", "Energy"])["abs_ypos"]
                    .to_dict()
                    )


# Map using (ADate, Energy)
sub_df["centre_abs_xpos"] = (
                                sub_df[["ADate", "Energy"]]
                                .apply(tuple, axis=1)
                                .map(centre_abs_xpos)
                                )

sub_df["centre_abs_ypos"] = (
                                sub_df[["ADate", "Energy"]]
                                .apply(tuple, axis=1)
                                .map(centre_abs_ypos)
                                )

sub_df["rel_xpos"] = sub_df['abs_xpos'] - sub_df["centre_abs_xpos"]
sub_df["rel_ypos"] = sub_df['abs_ypos'] - sub_df["centre_abs_ypos"]

## spot position plots
- abs_xpos, abs_ypos | tolerance = 2
- rel_xpos, rel_ypos | tolerance = 1

In [15]:
# plotting absolute y-pos, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last2 months
plot_by_spot_position(sub_df, "spot_position", "rel_ypos", "Gantry 2", "XRV-3000", 150, 0, 24, 1)

## fwhm data

In [16]:
# load fwhm reference dataset
# read data 
ref_fwhm_df = pd.read_excel(r"../data/xlsx_exported_from_access/ref/gantry_specific_baseline_2026_05_29.xlsx", sheet_name = "summary", header =0, usecols="A:E")
ref_fwhm_df["fwhm"] = ref_fwhm_df["fwhm"].round(4)


# commissioning data/ or the first annual QA data 
baseline = ref_fwhm_df[ref_fwhm_df["data_type"] =="baseline"]
baseline_ref = {}
for _, row in baseline.iterrows():
    baseline_ref.setdefault(row["gantry"], {}).setdefault(row["gantry_angle"], {})[row["energy"]] = row["fwhm"]

# tps data
tps = ref_fwhm_df[ref_fwhm_df["data_type"] =="tps"]
tps_ref = {}
for _, row in tps.iterrows():
    tps_ref.setdefault(row["gantry"], {}).setdefault(row["gantry_angle"], {})[row["energy"]] = row["fwhm"]

In [17]:
fwhm_df = df[['ADate', 'MachineName', 'Energy', 'Device', 'Gantry Angle', 'Spot', 'hor_fwhm', 'vert_fwhm','bltr_fwhm', 'tlbr_fwhm']].copy()
fwhm_df["ave_fwhw"] = fwhm_df[['hor_fwhm', 'vert_fwhm','bltr_fwhm', 'tlbr_fwhm']].mean(axis=1)


## fwhm plots
- 'hor_fwhm', 'vert_fwhm', 'bltr_fwhm', 'tlbr_fwhm', 'ave_fwhw'
- tps_ref[1][90][e] , e (int) = 70, 100, 150, 200,240
- baseline_ref[g][ga][e], g (int) = 1, 2, 3, 4 | ga (int) = 0, 90, 180, 240
- need to write a sentence - reference fwhm = average of horizontal and vertical fwhm

### Attention while building dash
- Using baseline_ref[g][ga][e], g (int) as argument for tolerance to plot fwhm data works as intended but increases the probability of errors during development since it's possible to plot the tolerances using measurements from alternate gantries/angles and energies instead of those associated with the FWHM data being shown. Recommend to unify arguments passed to prevent this.

In [22]:
# in callback function, we have to define g, ga, and e and pass these parameters to this function. 
gantry = "Gantry 2"
g = int(gantry.split(" ")[1])
ga = 0
e = 70  

plot_by_spot_position(fwhm_df, "fwhm", "hor_fwhm", gantry, "XRV-3000", e, ga, 24, baseline_ref[g][ga][e])

### Correction - option, parameter misalignment 
The function has been updated to validate the relationship between option and parameter, ensuring that only compatible combinations are accepted.
for example
- option = "spot_position" and parameter = "hor_fwhm"

In [29]:
gantry = "Gantry 2"
g = int(gantry.split(" ")[1])
ga = 0
e = 70  

plot_by_spot_position(fwhm_df, "spot_position", "hor_fwhm", gantry, "XRV-3000", e, ga, 24, baseline_ref[g][ga][e])

ValueError: Invalid parameter 'hor_fwhm' for option 'spot_position'. Expected one of ['abs_xpos', 'abs_ypos', 'rel_xpos', 'rel_ypos'].

## Gradient ratio data

In [19]:
gr_df = df[['ADate', 'MachineName', 'Energy', 'Device', 'Gantry Angle', 'Spot', 'hor_rt_gradient', 'hor_lt_gradient', 
       'vert_rt_gradient', 'vert_lt_gradient', 'bltr_rt_gradient','bltr_lt_gradient', 'tlbr_rt_gradient', 'tlbr_lt_gradient']].copy()

# gradient ratio (-1 is when the spot is perfectly symmetrical)
gr_df['gr_hor'] = gr_df["hor_rt_gradient"] / gr_df["hor_lt_gradient"] # horizontal gradient ratio
gr_df['gr_vert']  = gr_df["vert_rt_gradient"] / gr_df["vert_lt_gradient"] # vertical gradient ratio
gr_df['gr_bltr']  = gr_df["bltr_rt_gradient"] / gr_df["bltr_lt_gradient"] # bottom-left to top-right gradient ratio
gr_df['gr_tlbr']  = gr_df["tlbr_rt_gradient"] / gr_df["tlbr_lt_gradient"] # top-left to bottom-right gradient ratio

## Gradient ratio plots
- 'gr_hor', 'gr_vert', 'gr_bltr','gr_tlbr'
- tolerance  = None 

In [20]:
plot_by_spot_position(gr_df, "spot_symmetry", "gr_tlbr", "Gantry 2", "XRV-3000", 70, 0, 24, None)